Адаптируйте код `BaseMultimodalModel` так, чтобы слияние эмбеддингов происходило через конкатенацию. Выведите в ответе размерность получившегося вектора из прекода.

In [1]:
from transformers import AutoModelForCausalLM
from transformers import AutoModel, AutoTokenizer

import torch
import torch.nn as nn

import timm


class BaseMultimodalModel(nn.Module):
    def __init__(self,
                 text_model_name='bert-base-uncased',
                 image_model_name='resnet50',
                 emb_dim=256):
        super().__init__()
        self.emb_dim = emb_dim
        self.text_model = AutoModel.from_pretrained(text_model_name)
        self.image_model = timm.create_model(
            image_model_name,
            pretrained=True,
            num_classes=0 
        )

        self.text_proj = nn.Linear(self.text_model.config.hidden_size, emb_dim)
        self.image_proj = nn.Linear(self.image_model.num_features, emb_dim)

    def forward(self, text_input, image_input):
        text_features = self.text_model(**text_input).last_hidden_state[:,  0, :]
        image_features = self.image_model(image_input)

        text_emb = self.text_proj(text_features)
        image_emb = self.image_proj(image_features)

        fused_emb = torch.concat((text_emb, image_emb), dim=1)
        return fused_emb

text_model = 'bert-base-uncased'
image_model = 'resnet50'
m = BaseMultimodalModel(text_model_name=text_model,
                        image_model_name=image_model)

# 2 примера текста и картинки для инференса
tk = AutoTokenizer.from_pretrained(text_model)
tokenized = tk(["text", "text2"],
               return_tensors="pt",
               padding='max_length',
               truncation=True)

img = torch.randn(2, *m.image_model.pretrained_cfg["input_size"])
emb = m(tokenized, img)
emb.shape

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 4/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11202.37it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:


torch.Size([2, 512])

Добавьте в архитектуру `CrossAttentionModel` классификационный слой — двухслойный MLP с уменьшением размерности в два раза. Используйте дропаут и нормализации. Метод `forward()` должен возвращать логиты предсказаний. Количество классов `num_classes` установите равным 2. 


In [2]:
import torch.nn as nn


class CrossAttentionModel(nn.Module):
    def __init__(self, 
                 text_model_name='bert-base-uncased', 
                 image_model_name='resnet50', 
                 num_classes=2):
        super().__init__()
        self.base_model = BaseMultimodalModel(text_model_name, image_model_name)
        emb_dim = self.base_model.emb_dim
        
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=emb_dim*2, 
            num_heads=4)

        self.classifier = nn.Sequential(
            nn.Linear(emb_dim*2, emb_dim, bias=True),
            nn.LayerNorm(emb_dim),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(emb_dim, num_classes, bias=True),
        )
        
    def forward(self, text_input, image_input):
        text_emb, image_emb = self.base_model(text_input, image_input)
        
        text_emb = text_emb.unsqueeze(0)  
        image_emb = image_emb.unsqueeze(0)  
        
        attended, _ = self.cross_attn(
            query=text_emb,
            key=image_emb,
            value=image_emb
        )

        # логиты предсказаний
        logits = self.classifier(attended.squeeze(0))
        return logits

m = CrossAttentionModel(text_model_name=text_model,
                        image_model_name=image_model)

m(tokenized, img)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9735.76it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([-0.1302,  0.2011], grad_fn=<ViewBackward0>)